# Example Core Modes Sweep

Core transcendental functions (`exp..tanh`) across four JAX modes.

This notebook runs arbPlusJAX point/basic/adaptive/rigorous modes and compares to `c_chassis` where available.

In [ ]:
import os
import platform
import subprocess
import sys
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'arbplusjax').exists() and (p / 'tools').exists():
            return p
    raise RuntimeError(f'Could not locate repo root from: {start}')

REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

PYTHON = os.getenv('ARBPLUSJAX_PYTHON', sys.executable)
JAX_MODE = os.getenv('JAX_MODE', 'cpu').strip().lower()  # cpu or gpu
if JAX_MODE not in {'cpu', 'gpu'}:
    raise ValueError(f'JAX_MODE must be cpu or gpu, got: {JAX_MODE}')
JAX_DTYPE = os.getenv('JAX_DTYPE', 'float64').strip().lower()
if JAX_DTYPE not in {'float64', 'float32'}:
    raise ValueError(f'JAX_DTYPE must be float64 or float32, got: {JAX_DTYPE}')
SAMPLES = os.getenv('EXAMPLE_SAMPLES', '300,600')
SEEDS = os.getenv('EXAMPLE_SEEDS', '0,1')
C_REF_DIR = os.getenv('ARB_C_REF_DIR', str(REPO_ROOT / 'stuff' / 'migration' / 'c_chassis' / 'build_linux_wsl'))

def run(cmd: list[str], env_override: dict[str, str] | None = None):
    print('\n[cmd]', ' '.join(cmd))
    env = os.environ.copy()
    if env_override:
        env.update(env_override)
    cp = subprocess.run(cmd, cwd=REPO_ROOT, text=True, env=env)
    if cp.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}')

print('platform:', platform.system(), platform.release())
print('python:', PYTHON)
print('jax_mode:', JAX_MODE)
print('jax_dtype:', JAX_DTYPE)
print('samples:', SAMPLES)
print('seeds:', SEEDS)
print('c_ref_dir:', C_REF_DIR)

In [ ]:
env = os.environ.copy()
if JAX_MODE == 'gpu':
    run_env = {'JAX_PLATFORMS': 'cuda'}
    expected = 'gpu'
else:
    run_env = {'JAX_PLATFORMS': 'cpu'}
    expected = 'cpu'

run([PYTHON, 'tools/check_jax_runtime.py', '--expect-backend', expected], env_override=run_env)

In [ ]:
name = 'example_core_modes_' + JAX_MODE
cmd = [
    PYTHON, 'tools/run_harness_profile.py',
    '--name', name,
    '--functions', 'exp,log,sqrt,sin,cos,tan,sinh,cosh,tanh',
    '--samples', SAMPLES,
    '--seeds', SEEDS,
    '--jax-mode', JAX_MODE,
    '--jax-dtype', JAX_DTYPE,
    '--c-ref-dir', C_REF_DIR,
]
run(cmd)
OUT_DIR = REPO_ROOT / 'results' / 'benchmarks' / name
CSV_PATH = OUT_DIR / 'profile_summary.csv'
MD_PATH = OUT_DIR / 'profile_summary.md'
print('summary csv:', CSV_PATH)
print('summary md :', MD_PATH)

In [ ]:
df = pd.read_csv(CSV_PATH)
display(df.head(20))

num_cols = ['time_ms', 'mean_abs_err', 'containment_rate', 'peak_rss_mb']
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

agg = (
    df.groupby('backend', dropna=False)[['time_ms', 'mean_abs_err', 'containment_rate', 'peak_rss_mb']]
      .mean(numeric_only=True)
      .sort_values('time_ms', na_position='last')
)
display(agg)